Xây dựng công thức - Huấn luyện mô hình KMeans++

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score

df = pd.read_csv("product_data.csv")
X = df[['Price', 'Sales']].values

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def kmeans_plus_plus_init(X, k, random_state=0):
    np.random.seed(random_state)
    n_samples = X.shape[0]

    centroids = [X[np.random.randint(n_samples)]]

    for _ in range(k - 1):
        distances = np.zeros(n_samples)
        for i, point in enumerate(X):
            min_dist = float('inf')
            for centroid in centroids:
                dist = euclidean_distance(point, centroid)
                min_dist = min(min_dist, dist)
            distances[i] = min_dist

        distances_squared = distances ** 2
        probabilities = distances_squared / np.sum(distances_squared)

        next_centroid_idx = np.random.choice(n_samples, p=probabilities)
        centroids.append(X[next_centroid_idx])

    return np.array(centroids)

def assign_clusters(X, centroids):
    n_samples = X.shape[0]
    labels = np.zeros(n_samples, dtype=int)

    for i, point in enumerate(X):
        min_dist = float('inf')
        closest_centroid = 0

        for j, centroid in enumerate(centroids):
            dist = euclidean_distance(point, centroid)
            if dist < min_dist:
                min_dist = dist
                closest_centroid = j

        labels[i] = closest_centroid

    return labels

def update_centroids(X, labels, k):
    n_features = X.shape[1]
    centroids = np.zeros((k, n_features))

    for cluster_id in range(k):
        cluster_points = X[labels == cluster_id]
        if len(cluster_points) > 0:
            centroids[cluster_id] = np.mean(cluster_points, axis=0)

    return centroids

def kmeans_from_scratch(X, k, max_iters=300, random_state=0):
    centroids = kmeans_plus_plus_init(X, k, random_state)

    for iteration in range(max_iters):
        labels = assign_clusters(X, centroids)
        new_centroids = update_centroids(X, labels, k)

        if np.allclose(centroids, new_centroids):
            print(f"Hội tụ sau {iteration + 1} iterations")
            break

        centroids = new_centroids

    return labels, centroids

k = 2
labels, centroids = kmeans_from_scratch(X, k, random_state=0)

sil_score = silhouette_score(X, labels)
print(f"Silhouette Score: {sil_score:.2f}")

df['Cluster'] = labels

for cluster in range(k):
    print(f"\nCluster {cluster}:")
    cluster_products = df[df['Cluster'] == cluster].head(12)
    print(cluster_products['Product'])

Hội tụ sau 2 iterations
Silhouette Score: 0.74

Cluster 0:
0      Product 1
1      Product 2
2      Product 3
3      Product 4
4      Product 5
5      Product 6
6      Product 7
7      Product 8
8      Product 9
9     Product 10
10    Product 11
11    Product 12
Name: Product, dtype: object

Cluster 1:
200    Product 201
201    Product 202
202    Product 203
203    Product 204
204    Product 205
205    Product 206
206    Product 207
207    Product 208
208    Product 209
209    Product 210
210    Product 211
211    Product 212
Name: Product, dtype: object


Sử dụng thư viện - Huấn luyện mô hình KMeans++

In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("product_data.csv")
X = df[['Price', 'Sales']].values

k = 2
kmeans = KMeans(n_clusters=k,init = "k-means++", random_state=0)
labels = kmeans.fit_predict(X)

sil_score = silhouette_score(X, labels)
print(f"Silhouette Score: {sil_score:.2f}")
df['Cluster'] = labels

for cluster in range(k):
    print(f"\nCluster {cluster}:")
    cluster_products = df[df['Cluster'] == cluster].head(12)
    print(cluster_products['Product'])

Silhouette Score: 0.74

Cluster 0:
0      Product 1
1      Product 2
2      Product 3
3      Product 4
4      Product 5
5      Product 6
6      Product 7
7      Product 8
8      Product 9
9     Product 10
10    Product 11
11    Product 12
Name: Product, dtype: object

Cluster 1:
200    Product 201
201    Product 202
202    Product 203
203    Product 204
204    Product 205
205    Product 206
206    Product 207
207    Product 208
208    Product 209
209    Product 210
210    Product 211
211    Product 212
Name: Product, dtype: object
